
# SciLLM × Chutes: Complete Call Examples

Each section below shows a fully expanded SciLLM call for a specific scenario, followed by one combined batch example. Copy any code cell into your own script to reproduce the behavior.



Referenced source files for Copilot/code reviewers:
- `scripts/sanity/chutes_batch_sanity.py` – canonical probe definitions + summary checks
- `scillm/batch.py` – `parallel_acompletions_iter`, tenacity, timeout/backoff handling
- `scillm/preprocess.py` – request I/O expansion, inline asset handling
- `scillm/extras/json_utils.py` – `clean_json_string` fallback used when JSON repair is enabled



## Environment + Imports
- Install deps: `uv pip install -e .[scillm]`
- `.env` must define `CHUTES_API_BASE`, `CHUTES_API_KEY`, `CHUTES_TEXT_MODEL` (or `CHUTES_MODEL_ID`), `CHUTES_VLM_MODEL`
- Assets: `scripts/sanity/assets/inline_classification.html`, `docs/assets/screenshots/220px-Giant_Panda_Tai_Shan.JPG`


In [ ]:

import asyncio
import json
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from scillm import parallel_acompletions_iter
from scillm.batch import _extract_content_from_response
from scillm.extras.json_utils import clean_json_string


In [ ]:

load_dotenv(find_dotenv(), override=False)

REPO_ROOT = Path.cwd()
HTML_FIXTURE = REPO_ROOT / "scripts" / "sanity" / "assets" / "inline_classification.html"
IMAGE_FIXTURE = REPO_ROOT / "docs" / "assets" / "screenshots" / "220px-Giant_Panda_Tai_Shan.JPG"
REMOTE_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0f/Grosser_Panda.JPG/2560px-Grosser_Panda.JPG"
assert HTML_FIXTURE.exists(), HTML_FIXTURE
assert IMAGE_FIXTURE.exists(), IMAGE_FIXTURE


In [ ]:

base = os.getenv("CHUTES_API_BASE", "").rstrip("/")
key = os.getenv("CHUTES_API_KEY")
text_model = os.getenv("CHUTES_TEXT_MODEL") or os.getenv("CHUTES_MODEL_ID")
vlm_model = os.getenv("CHUTES_VLM_MODEL")
required = {
    "CHUTES_API_BASE": base,
    "CHUTES_API_KEY": key,
    "CHUTES_TEXT_MODEL or CHUTES_MODEL_ID": text_model,
    "CHUTES_VLM_MODEL": vlm_model,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise RuntimeError(f"Missing required environment variables: {missing}")

MODEL_LIST = [
    {
        "model_name": "chutes/text",
        "litellm_params": {
            "custom_llm_provider": "openai_like",
            "model": text_model,
            "api_base": base,
            "api_key": key,
        },
    },
    {
        "model_name": "chutes/vlm",
        "litellm_params": {
            "custom_llm_provider": "openai_like",
            "model": vlm_model,
            "api_base": base,
            "api_key": key,
        },
    },
]
TEXT_SLOT = "chutes/text"
VLM_SLOT = "chutes/vlm"


In [ ]:

os.environ.setdefault("SCILLM_AUTO_IMAGE_DATAURL", "1")

async def _call_chutes_once(
    request: dict,
    *,
    concurrency: int = 1,
    request_timeout_s: float = 45,
    retry_wall_time_s: float = 120,
    retry_initial_delay: float = 0.5,
    retry_max_delay: float = 30,
    tenacious: bool = True,
    inline_remote_images: bool = True,
    json_sanitize: bool = True,
):
    if inline_remote_images:
        os.environ["SCILLM_INLINE_REMOTE_IMAGES"] = "1"
    results = []
    async for entry in parallel_acompletions_iter(
        [request],
        model_list=MODEL_LIST,
        concurrency=concurrency,
        wall_time_s=retry_wall_time_s,
        timeout=request_timeout_s,
        tenacious=tenacious,
        backoff_base=retry_initial_delay,
        backoff_cap_s=retry_max_delay,
    ):
        results.append(entry)
    if not results:
        raise RuntimeError("No response returned")
    entry = results[0]
    content = entry.get("content")
    if content is None:
        content = _extract_content_from_response(entry.get("response"))
    parsed = None
    if isinstance(content, str):
        try:
            parsed = json.loads(content.strip())
        except Exception:
            if json_sanitize:
                parsed = clean_json_string(content, return_dict=True)
    elif isinstance(content, (dict, list)):
        parsed = content
    return {
        "ok": bool(entry.get("ok")) and not entry.get("error"),
        "error": entry.get("error"),
        "raw_content": content,
        "parsed": parsed,
        "model_used": request.get("model"),
    }


def run_single_request(request: dict, **overrides):
    return asyncio.run(_call_chutes_once(request, **overrides))



### Example 1 — Text JSON echo (`json_probe`)
Goal: confirm the text slot responds with `{ "ok": true }` when `response_format` is set.


In [ ]:

json_probe_request = {
    "model": TEXT_SLOT,
    "messages": [
        {"role": "system", "content": "Only respond in well formatted JSON"},
        {"role": "user", "content": "Return only {"ok":true} as JSON."},
    ],
    "response_format": {"type": "json_object"},
    "max_tokens": 16,
    "temperature": 0,
}
print("REQUEST:")
print(json.dumps(json_probe_request, indent=2))
json_probe_result = run_single_request(json_probe_request)
print("
RESPONSE:")
print(json.dumps(json_probe_result, indent=2))



### Example 2 — Structured QA (`france_capital`)
Goal: return `{country, capital}` strictly as JSON.


In [ ]:

france_capital_request = {
    "model": TEXT_SLOT,
    "messages": [
        {"role": "system", "content": "Only respond in well formatted JSON"},
        {
            "role": "user",
            "content": "What is the capital of France? Respond with {country:<string>, capital:<string>} strictly as JSON.",
        },
    ],
    "response_format": {"type": "json_object"},
    "max_tokens": 32,
    "temperature": 0,
}
print("REQUEST:")
print(json.dumps(france_capital_request, indent=2))
france_capital_result = run_single_request(france_capital_request)
print("
RESPONSE:")
print(json.dumps(france_capital_result, indent=2))



### Example 3 — Vision via HTTPS URL (`vlm_https_image`)
Goal: describe a remote panda photo. When `inline_remote_images=True`, SciLLM downloads and base64-encodes the JPEG before sending it to Chutes.


In [ ]:

vlm_https_request = {
    "model": VLM_SLOT,
    "messages": [
        {"role": "system", "content": "Only respond in well formatted JSON"},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe the following image as {"description":<string>} strictly as JSON."},
                {"type": "image_url", "image_url": {"url": REMOTE_IMAGE_URL}},
            ],
        },
    ],
    "response_format": {"type": "json_object"},
    "max_tokens": 128,
    "temperature": 0.2,
    "artifacts": {"file_paths": [], "urls": [REMOTE_IMAGE_URL]},
}
print("REQUEST:")
print(json.dumps(vlm_https_request, indent=2))
vlm_https_result = run_single_request(vlm_https_request, inline_remote_images=True)
print("
RESPONSE:")
print(json.dumps(vlm_https_result, indent=2))



### Example 4 — Vision via local file (`vlm_file_image`)
Goal: describe a checked-in panda image by referencing its file path. SciLLM auto-inlines the bytes because `SCILLM_AUTO_IMAGE_DATAURL=1`.


In [ ]:

vlm_file_request = {
    "model": VLM_SLOT,
    "file_path": str(IMAGE_FIXTURE),
    "messages": [
        {"role": "system", "content": "Only respond in well formatted JSON"},
        {
            "role": "user",
            "content": f"Describe the local image located at {IMAGE_FIXTURE} as {{"description":<string>}} strictly as JSON.",
        },
    ],
    "response_format": {"type": "json_object"},
    "max_tokens": 128,
    "temperature": 0.2,
    "artifacts": {"file_paths": [str(IMAGE_FIXTURE)], "urls": []},
}
print("REQUEST:")
print(json.dumps(vlm_file_request, indent=2))
vlm_file_result = run_single_request(vlm_file_request)
print("
RESPONSE:")
print(json.dumps(vlm_file_result, indent=2))



### Example 5 — HTML classification (`html_inline_classification`)
Goal: read a local HTML file and return the page's `Classification Label` value as JSON.


In [ ]:
html_classification_request = {
    "model": TEXT_SLOT,
    "file_path": str(HTML_FIXTURE),
    "messages": [
        {"role": "system", "content": "Only respond in well formatted JSON"},
        {
            "role": "user",
            "content": f"Read the provided HTML document located at {HTML_FIXTURE} (appended separately) and return {\"category\":<string>} using the exact 'Classification Label' value.",
        },
    ],
    "response_format": {"type": "json_object"},
    "max_tokens": 16,
    "temperature": 0,
    "artifacts": {"file_paths": [str(HTML_FIXTURE)], "urls": []},
}
print("REQUEST:")
print(json.dumps(html_classification_request, indent=2))
html_classification_result = run_single_request(html_classification_request)
print("
RESPONSE:")
print(json.dumps(html_classification_result, indent=2))



## Combined batch example
Bundle all five requests and run them together. This mirrors the CLI sanity script while still streaming each completion via `parallel_acompletions_iter`.


In [ ]:

batch_requests = [
    json_probe_request,
    france_capital_request,
    vlm_https_request,
    vlm_file_request,
    html_classification_request,
]

async def run_batch(requests):
    collected = []
    async for entry in parallel_acompletions_iter(
        requests,
        model_list=MODEL_LIST,
        concurrency=3,
        wall_time_s=120,
        timeout=45,
        tenacious=True,
        backoff_base=0.5,
        backoff_cap_s=30,
    ):
        collected.append(entry)
    return collected

batch_results = asyncio.run(run_batch(batch_requests))
print(json.dumps(batch_results, indent=2))


In [ ]:
import time
HTML_LABEL = "luminous-harvest"
scenarios = [
    "json_probe",
    "france_capital",
    "vlm_https_image",
    "vlm_file_image",
    "html_inline_classification",
]
start = time.time()
batch_results = asyncio.run(run_batch(batch_requests))
items = []
all_ok = True
success_count = 0
failure_count = 0
ordered = sorted(batch_results, key=lambda e: e.get("index", 0))
for idx, entry in enumerate(ordered):
    req = entry.get("request") or batch_requests[idx]
    raw_content = entry.get("content")
    ok = bool(entry.get("ok")) and not entry.get("error")
    reason = None
    parsed = None
    content = ""
    if ok:
        if isinstance(raw_content, str):
            content = raw_content.strip()
            try:
                parsed = json.loads(content)
                ok = isinstance(parsed, (dict, list)) and len(json.dumps(parsed)) > 0
            except Exception:
                try:
                    parsed_candidate = clean_json_string(content, return_dict=True)
                except Exception:
                    parsed_candidate = None
                if isinstance(parsed_candidate, (dict, list)) and len(json.dumps(parsed_candidate)) > 0:
                    parsed = parsed_candidate
                    content = json.dumps(parsed_candidate)
                    ok = True
                else:
                    ok = False
                    reason = "invalid_json:sanitize_failed"
        elif isinstance(raw_content, (dict, list)):
            parsed = raw_content
            content = json.dumps(raw_content)
            ok = True
        elif raw_content is None:
            ok = False
            reason = "empty_content"
        else:
            content = str(raw_content)
            try:
                parsed = json.loads(content)
                ok = isinstance(parsed, (dict, list)) and len(json.dumps(parsed)) > 0
            except Exception as e:
                ok = False
                reason = f"invalid_json:{e}"
    else:
        reason = entry.get("error") or "unknown_error"
    scenario = scenarios[idx]
    if ok and isinstance(parsed, dict):
        if scenario == "france_capital":
            ctry = str(parsed.get("country") or "").lower()
            cap = str(parsed.get("capital") or "").lower()
            if not (ctry == "france" and cap == "paris"):
                ok = False
                reason = f"mismatch:country={ctry},capital={cap}"
        if scenario in {"vlm_https_image", "vlm_file_image"}:
            desc = parsed.get("description")
            if not isinstance(desc, str) or not desc.strip():
                ok = False
                reason = "no_description"
        if scenario == "html_inline_classification":
            category = str(parsed.get("category") or "").strip().lower()
            if category != HTML_LABEL:
                ok = False
                reason = f"mismatch:category={category}"
        if scenario == "json_probe":
            if parsed.get("ok") is not True:
                ok = False
                reason = "ok_flag_missing"
    items.append({
        "index": idx,
        "scenario": scenario,
        "ok": ok,
        "reason": reason,
        "content_head": (content or "")[:160].replace("
", " "),
        "model_used": req.get("model"),
        "artifacts": req.get("artifacts") or {},
    })
    if ok:
        success_count += 1
    else:
        failure_count += 1
    all_ok = all_ok and ok
elapsed = round(time.time() - start, 3)
summary = {
    "ok": all_ok,
    "count": len(items),
    "items": items,
    "tenacious": True,
    "error": None,
    "elapsed_s": elapsed,
    "success_count": success_count,
    "failure_count": failure_count,
}
print(json.dumps(summary, ensure_ascii=False))
print(f"SUMMARY chutes_batch_sanity ok={1 if summary['ok'] else 0} count={len(items)} success={success_count} failure={failure_count} elapsed_s={elapsed}")